In [1]:
# SOCIAL TREND ANALYSIS - MULTILINGUAL EMOTION COMPARISON
# Compare text emotions across decades for English, Hindi, and Tamil

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import os
from scipy import stats

warnings.filterwarnings("ignore")

EMOTION_LABELS = ["anger", "contempt", "disgust", "fear", "frustration", 
                "gratitude", "joy", "love", "neutral", "sadness", "surprise"]

print("="*60)
print("SOCIAL TREND ANALYSIS - MULTILINGUAL")
print("="*60)

SOCIAL TREND ANALYSIS - MULTILINGUAL


In [2]:
# Load all emotion datasets
english_df = pd.read_csv("final_english_emotions.csv") if os.path.exists("final_english_emotions.csv") else None
hindi_df = pd.read_csv("final_hindi_emotions.csv") if os.path.exists("final_hindi_emotions.csv") else None
tamil_df = pd.read_csv("final_tamil_emotions.csv") if os.path.exists("final_tamil_emotions.csv") else None

# Standardize year column
def standardize_year(df):
    if df is None:
        return None
    df = df.copy()
    if 'time_period' in df.columns:
        df['year'] = df['time_period'].astype(str).str.replace('s', '').astype(int)
    return df

english_df = standardize_year(english_df)
hindi_df = standardize_year(hindi_df)
tamil_df = standardize_year(tamil_df)

print(f"\nLoaded datasets:")
print(f"  English: {len(english_df) if english_df is not None else 0} rows")
print(f"  Hindi: {len(hindi_df) if hindi_df is not None else 0} rows")
print(f"  Tamil: {len(tamil_df) if tamil_df is not None else 0} rows")


Loaded datasets:
  English: 24403 rows
  Hindi: 4705 rows
  Tamil: 3455 rows


In [3]:
# Aggregate by decade
def aggregate_by_decade(df, language):
    if df is None or len(df) == 0:
        return None
    
    df = df.copy()
    df['decade'] = (df['year'] // 10) * 10
    
    # Calculate sentiment scores
    positive_cols = ['gratitude', 'joy', 'love']
    negative_cols = ['anger', 'contempt', 'disgust', 'fear', 'frustration', 'sadness']
    
    df['positive'] = df[positive_cols].mean(axis=1)
    df['negative'] = df[negative_cols].mean(axis=1)
    df['net_sentiment'] = df['positive'] - df['negative']
    
    # Aggregate
    agg_dict = { emotion: 'mean' for emotion in EMOTION_LABELS }
    agg_dict.update({'positive': 'mean', 'negative': 'mean', 'net_sentiment': 'mean', 'year': 'count'})
    
    result = df.groupby('decade').agg(agg_dict).reset_index()
    result.rename(columns={'year': 'count'}, inplace=True)
    result['language'] = language
    return result

# Aggregate each language
english_decade = aggregate_by_decade(english_df, 'English') if english_df is not None else None
hindi_decade = aggregate_by_decade(hindi_df, 'Hindi') if hindi_df is not None else None
tamil_decade = aggregate_by_decade(tamil_df, 'Tamil') if tamil_df is not None else None

print("\n" + "="*60)
print("AGGREGATED BY DECADE")
print("="*60)


AGGREGATED BY DECADE


In [4]:
# Display aggregated data
if english_decade is not None:
    print("\n--- ENGLISH ---")
    print(english_decade[['decade', 'positive', 'negative', 'net_sentiment', 'count']].to_string(index=False))

if hindi_decade is not None:
    print("\n--- HINDI ---")
    print(hindi_decade[['decade', 'positive', 'negative', 'net_sentiment', 'count']].to_string(index=False))

if tamil_decade is not None:
    print("\n--- TAMIL ---")
    print(tamil_decade[['decade', 'positive', 'negative', 'net_sentiment', 'count']].to_string(index=False))


--- ENGLISH ---
 decade  positive  negative  net_sentiment  count
   1810  0.082258  0.070477       0.011781   1609
   1820  0.089577  0.073636       0.015941   1420
   1830  0.065274  0.074225      -0.008951   1320
   1840  0.072071  0.078344      -0.006273   1813
   1850  0.060150  0.063562      -0.003412   2220
   1860  0.070739  0.063549       0.007190   2400
   1870  0.046782  0.062759      -0.015977   2040
   1880  0.056922  0.067845      -0.010923   2340
   1890  0.050762  0.058720      -0.007959   2220
   1900  0.054182  0.064774      -0.010592   2200
   1910  0.056767  0.056046       0.000720   1395
   1920  0.050755  0.055694      -0.004939   2123
   1930  0.042100  0.063090      -0.020990    583
   1940  0.035278  0.046424      -0.011146    360
   1950  0.036917  0.058495      -0.021578    240
   1960  0.087589  0.034631       0.052958    120

--- HINDI ---
 decade  positive  negative  net_sentiment  count
   1810  0.034100  0.034054       0.000047     71
   1820  0.018480 

In [5]:
# Compare sentiment across languages
fig = make_subplots(rows=1, cols=3, 
    subplot_titles=("English", "Hindi", "Tamil"),
    shared_yaxes=True)

colors = {'English': 'blue', 'Hindi': 'orange', 'Tamil': 'green'}

if english_decade is not None:
    fig.add_trace(go.Scatter(
        x=english_decade['decade'], y=english_decade['net_sentiment'],
        mode='lines+markers', name='English',
        marker=dict(color='blue'), line=dict(color='blue')
    ), row=1, col=1)

if hindi_decade is not None:
    fig.add_trace(go.Scatter(
        x=hindi_decade['decade'], y=hindi_decade['net_sentiment'],
        mode='lines+markers', name='Hindi',
        marker=dict(color='orange'), line=dict(color='orange')
    ), row=1, col=2)

if tamil_decade is not None:
    fig.add_trace(go.Scatter(
        x=tamil_decade['decade'], y=tamil_decade['net_sentiment'],
        mode='lines+markers', name='Tamil',
        marker=dict(color='green'), line=dict(color='green')
    ), row=1, col=3)

fig.update_layout(
    title='Net Sentiment by Decade (Positive - Negative)',
    height=400, template='plotly_white',
    showlegend=True
)
fig.update_yaxes(title_text='Net Sentiment')
fig.show()

In [6]:
# Heatmap of emotions by decade for each language
def plot_emotion_heatmap(decade_df, title):
    if decade_df is None:
        return
    
    fig = go.Figure(data=go.Heatmap(
        z=decade_df[EMOTION_LABELS].T,
        x=decade_df['decade'],
        y=EMOTION_LABELS,
        colorscale='RdBu',
        zmid=0.5
    ))
    fig.update_layout(title=f'{title}: Emotion Scores by Decade', height=400)
    fig.show()

if english_decade is not None:
    plot_emotion_heatmap(english_decade, 'English')
if hindi_decade is not None:
    plot_emotion_heatmap(hindi_decade, 'Hindi')
if tamil_decade is not None:
    plot_emotion_heatmap(tamil_decade, 'Tamil')

In [7]:
# Combined comparison chart
fig = go.Figure()

if english_decade is not None:
    fig.add_trace(go.Bar(
        x=english_decade['decade'], y=english_decade['positive'],
        name='English Positive', marker_color='blue'
    ))
    fig.add_trace(go.Bar(
        x=english_decade['decade'], y=-english_decade['negative'],
        name='English Negative', marker_color='lightblue'
    ))

if hindi_decade is not None:
    fig.add_trace(go.Bar(
        x=hindi_decade['decade'], y=hindi_decade['positive'],
        name='Hindi Positive', marker_color='orange'
    ))
    fig.add_trace(go.Bar(
        x=hindi_decade['decade'], y=-hindi_decade['negative'],
        name='Hindi Negative', marker_color='wheat'
    ))

if tamil_decade is not None:
    fig.add_trace(go.Bar(
        x=tamil_decade['decade'], y=tamil_decade['positive'],
        name='Tamil Positive', marker_color='green'
    ))
    fig.add_trace(go.Bar(
        x=tamil_decade['decade'], y=-tamil_decade['negative'],
        name='Tamil Negative', marker_color='lightgreen'
    ))

fig.update_layout(
    barmode='overlay',
    title='Positive vs Negative Sentiment by Decade',
    height=500, template='plotly_white',
    yaxis_title='Sentiment Score'
)
fig.show()

In [8]:
# Export combined data
dfs = []
if hindi_decade is not None:
    dfs.append(hindi_decade)
if tamil_decade is not None:
    dfs.append(tamil_decade)
if english_decade is not None:
    dfs.append(english_decade)

if dfs:
    combined = pd.concat(dfs, ignore_index=True)
    combined = combined.sort_values(['language', 'decade'])
    combined.to_csv("social_trend_analysis.csv", index=False)
    print(f"\nExported to social_trend_analysis.csv")
    print(f"Total rows: {len(combined)}")
    print(f"Languages: {combined['language'].unique().tolist()}")
    print(f"Decades: {sorted(combined['decade'].unique().tolist())}")


Exported to social_trend_analysis.csv
Total rows: 48
Languages: ['English', 'Hindi', 'Tamil']
Decades: [1810, 1820, 1830, 1840, 1850, 1860, 1870, 1880, 1890, 1900, 1910, 1920, 1930, 1940, 1950, 1960, 1970, 1980]
